In [ ]:
from platform import python_version
print(python_version())

## Spatiotemporal gene expression and cellular dynamics of the developing human heart (forcked)

This repository contains the scripts used for the analysis of the single-cell RNA seq and the Spatial Transcriptomics data of the embryonic heart developmental atlas (HDCA). 
Since several environments have been used along the project, below you will find a workflow chart presenting the pipeline used with the corresponding scripts. 

## Source and Data

- Github: https://github.com/rmauron/HDCA_heart_dev
  - Code: https://github.com/rmauron/HDCA_heart_dev/tree/main/code
  - Environment: https://github.com/rmauron/HDCA_heart_dev/tree/main/environments
    - different yml
      - liana (biocondutorSS)
      - scFates
      - scVelo
      - stereoscope

- Zenodo: https://zenodo.org/records/15912657

- Mendeley: https://data.mendeley.com/datasets
  - /bundle.js?342865b28b070babe69b




### Other tutorials and studies


Current Best Practices in Single-Cell RNA-Seq Analysis: A Tutorial [Luecken & Theis, 2019](https://doi.org/10.15252/msb.20188746) by Malte Lücken and Fabian Theis introduces best practices for scRNA-seq analysis. Its key contribution lies in not only reviewing potential analysis steps but also recommending best practices based on independent benchmarks. When specific best-practice guidelines are unavailable, the authors provide general recommendations for analysis approaches. The fundamental idea of focusing on independent benchmarks inspired our work substantially. The paper is complemented by [an example analysis of mouse intestinal epithelium regions](https://github.com/theislab/single-cell-tutorial/) from Haber et al. [Haber et al., 2017](https://doi.org/10.1038/nature24489).


### Scripts for "Current best-practices in single-cell RNA-seq: a tutorial"

https://github.com/theislab/single-cell-tutorial/

![workflow](../figures/singlecell_workflow.png)

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/HDCA_heart_dev")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

ROOT0_DATA = create_dir(ROOT0, "data")
root_colab = create_dir(ROOT0_DATA, "colab")

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)


### Scanpy Introduction

https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#

In [ ]:
# Core scverse libraries
from __future__ import annotations

import anndata as ad

# Pooch is a Python library that can manage data by downloading files from a server (only when needed) and storing them locally in a data cache 
import pooch
import scanpy as sc
import squidpy as sq
import importlib

# scanpy 1.12.4 | squidpy 1.8.3 | anndata 0.13.3.post0
import anndata
import json

importlib.metadata.version("scanpy"), importlib.metadata.version("squidpy"), importlib.metadata.version("anndata")


In [ ]:
sc.set_figure_params(dpi=120, facecolor="white")

### Getting data

The data used in this basic preprocessing and clustering tutorial was collected from bone marrow mononuclear cells of healthy human donors and was part of openproblem’s NeurIPS 2021 benchmarking dataset [Luecken et al., 2021]. The samples used in this tutorial were measured using the 10X Multiome Gene Expression and Chromatin Accessability kit.

We are reading in the count matrix into an AnnData object, which holds many slots for annotations and different representations of the data. See Getting started with anndata for a tutorial.

anndata: https://anndata.scverse.org/en/stable/tutorials/notebooks/getting-started.html

In [ ]:
root_h5 = create_dir(ROOT0_DATA, "V10A13-157_A1")
root_spatial = create_dir(root_h5, 'spatial')
os.listdir(root_h5)

In [ ]:
root_meta = create_dir(ROOT0_DATA, "meta")
os.listdir(root_meta)

### what is the content in the files? 

- /data/V10A13-157_A1/
  - **filtered_feature_bc_matrix.h5** (5.96 MB)
    - "Filtered" means spots under tissue only — the 1,070 count already excludes off-tissue background. No coordinates, no image, no QC, no annotation: those live in the sibling spatial/ directory.
    - Space Ranger 1.2.1 output, CellRanger HDF5 v2 format, chemistry "Spatial 3' v1". Raw UMI counts, nothing else.
      - 36,601 genes × 1,070 spots, stored CSC-style (data/indices/indptr)
      - 4,041,097 non-zeros — 10.3% dense; 13,696,700 total UMI
        - Integer counts, range 1–1,636
        - Features: all Gene Expression, genome GRCh38, with Ensembl IDs (ENSG00000243485) and symbols (MIR1302-2HG) — no antibody/CRISPR capture
        - Barcodes are Visium spot barcodes (AAACATTTCCCGGATT-1)


  - **V10A13-157_A1_processed.h5ad** (291 MB)
    - Full analysis output. 1,067 spots × 17,673 genes — three spots and ~19k genes dropped by QC filtering.
    - Three copies of the expression data at different processing stages:

| Slot | Content | Type | On disk | Use for |
|---|---|---|---|---|
| `layers['counts']` | raw UMI, max 1,636 | sparse float32 | 32.3 MB | re-normalising, count-based models |
| `.raw.X` | log1p-normalised (`expm1` per spot = 10,000) | sparse float32 | 32.3 MB | plotting, DE testing |
| `.X` | z-scored, clipped ±10 | dense float64 | 150.9 MB | PCA input only |

  - That dense .X is 52% of the file. Use .raw.X for any expression plotting or differential testing; .X is only appropriate as PCA input.
    - obs (31 columns): QC metrics (total_counts, pct_counts_mt, ribo, hb); Leiden at four resolutions (0.3/0.5/0.8/1.0); eight sc.tl.score_genes signature scores; spatial geometry (dist_surface_um, depth_lat, n_nb, is_edge); and the final domain/domain_short labels.

### Leiden

The six domains, from leiden_0.5:
  - Compact ventricular myocardium 343, 
  - Trabecular ventricular myocardium 265, 
  - Fibrous–interstitial myocardium 154, 
  - Left atrial myocardium 119, 
  - Right atrial myocardium 107, 
  - Valve mesenchyme 
  - Vessel wall 79.


### H&E images

uns: the H&E images (hires 1937×2000×3, lowres 581×600×3) plus scalefactors — 50.7 MB; moranI (2,000 HVGs × 4); rank_genes_groups; c0_vs_c4 (the LA-vs-RA test); colour palettes.


### PCA - UMAP

obsm/obsp: PCA (50 comps), UMAP, spatial coordinates; expression k-NN graph (23,820 edges) and a separate spatial neighbour graph (5,176 edges).

One caveat on the marker scores. The mean signature scores confirm what I reported earlier: Compact vCM 0.879, Trabecular vCM 0.775, Atrial CM 1.072 are strongly positive, but V_EP (−0.010), EN (−0.016), AVP (0.001), and VM (0.011) sit at zero. Epicardium, endocardium, and the AV plane are not resolved at 55 µm spot resolution — those four columns are present in the file but carry no usable signal. Don't interpret them as evidence of absence; they're a resolution limit.

If you want that dense .X dropped to get the file to ~140 MB, it's one line — everything downstream can be rebuilt from .raw.X and the stored PCs.


In [ ]:
open(root_spatial / "scalefactors_json.json").read()

In [ ]:

fname = "tissue_positions_list.csv"
filename = root_spatial / fname

df_tp=pd.read_csv(filename, header=None)
print(df_tp.shape)
df_tp.head(3)


In [ ]:
import shutil
from PIL import Image

Image.MAX_IMAGE_PIXELS=None

In [ ]:
root_spatial

In [ ]:
# Space Ranger / scanpy expects a lowres image too; derive it from hires using scalefactors
filename = root_spatial / "scalefactors_json.json"

if filename.exists():
    sf=json.load(open(filename))
    ratio=sf["tissue_lowres_scalef"]/sf["tissue_hires_scalef"]
else:
    print(f"Scalefactors file not found: {filename}.")
    ratio=-1

ratio

In [ ]:
from IPython.display import display

hi=Image.open(root_spatial / "tissue_hires_image.png")

lo=hi.resize((int(hi.width*ratio), int(hi.height*ratio)), Image.LANCZOS)
lo.save(root_spatial / "tissue_lowres_image.png")
print("hires",hi.size,"-> lowres",lo.size,"ratio",round(ratio,4))

hi.show()
display(lo)

In [ ]:
root_h5

### Reding Visium with Squidpy (sq)

var_names_make_unique() is still required — squidpy doesn't do it for you, and this reference has duplicate gene symbols (the UserWarning fires on both readers). Keep that line.

load_images=False is worth knowing about if you batch the PCW 5.5–14 series: the H&E occupies 50.7 MB inside a single processed .h5ad, so skipping images across 34 sections saves real memory when you only need counts and coordinates.

In [ ]:

SEC="V10A13-157_A1"
fname_h5 = "filtered_feature_bc_matrix.h5"
filename = root_h5 / fname_h5

if filename.exists():
    ad = sq.read.visium(root_h5, counts_file=fname_h5, library_id=SEC, load_images=True,) 
    ad.var_names_make_unique()
else:
    print(f"h5 file not found: {filename}.")
    ad = None

print(ad)

In [ ]:
print("obs cols ad:", list(ad.obs.columns))
print("uns spatial keys ad:", list(ad.uns["spatial"][SEC]))
print("images ad:", {k:v.shape for k,v in ad.uns["spatial"][SEC]["images"].items()})
print("metadata ad:", ad.uns["spatial"][SEC].get("metadata"))

### Quality control (QC)

#### calculate_qc_metrics

Calculate quality control metrics.

Calculates a number of qc metrics for an AnnData object, see section Returns for specifics. Largely based on calculateQCMetrics from scater [McCarthy et al., 2017]. Currently is most efficient on a sparse CSR or dense matrix.

https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html#scanpy.pp.calculate_qc_metrics

qc_vars: Keys for boolean columns of .var which identify variables you could want to control for (e.g. “ERCC” or “mito”).

In [ ]:
ad.var_names.shape, ad.var_names[:5]

In [ ]:
ad.var["mt"] = ad.var_names.str.startswith("MT-")
ad.var["ribo"] = ad.var_names.str.startswith(("RPS","RPL"))
ad.var["hb"] = ad.var_names.str.contains(r"^HB[AB]")
sc.pp.calculate_qc_metrics(ad, qc_vars=["mt","ribo","hb"], inplace=True, log1p=False, percent_top=None)

print("in_tissue:", ad.obs.in_tissue.value_counts().to_dict())
q = ad.obs[["total_counts","n_genes_by_counts","pct_counts_mt","pct_counts_ribo","pct_counts_hb"]]
print(q.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).round(2).to_string())
print("\ngenes detected in >=1 spot:", int((ad.var.n_cells_by_counts>0).sum()), "of", ad.n_vars)
print("spots <500 counts:", int((ad.obs.total_counts<500).sum()), "| <250 genes:", int((ad.obs.n_genes_by_counts<250).sum()), "| mt>20%:", int((ad.obs.pct_counts_mt>20).sum()))


### One can now inspect violin plots of some of the computed QC metrics:

- the number of genes expressed in the count matrix
- the total counts per cell
- the percentage of counts in mitochondrial genes


### These are the three standard per-cell QC metrics that sc.pp.calculate_qc_metrics writes into adata.obs. 

Each is computed per cell (one value per row/barcode):

- total_counts — the sum of all UMI counts across every gene in that cell. 
  - This is the library size / sequencing depth for the cell. 
  - Also called total counts or nUMI. 
  - A cell with total_counts = 12000 had 12,000 transcripts captured.

- n_genes_by_counts — the number of distinct genes with at least one count in that cell (i.e., how many genes are non-zero). 
  - This measures transcriptional complexity. A cell can have high total counts but few genes if a handful of genes dominate.

What each fraction tells you

- pct_counts_mt - mitochondrial fraction. 
  - High values flag dying or lysing cells: the membrane ruptured, cytoplasmic mRNA leaked out, and the mito transcripts protected inside mitochondria stayed behind, so their proportion spikes. Classic apoptosis/stress signature.

- pct_counts_ribo - ribosomal-protein fraction. 
  - Ribosomal genes are among the most highly expressed transcripts in any cell, so this is often 20–50% and that alone isn't a problem. 
  - It's used more as a covariate and a cell-type signal than a hard filter: 
    - proliferating and metabolically active cells (relevant in tumors) run high, 
    - and abnormally low ribo can also mark low-quality cells.
  - Some pipelines regress it out or exclude ribo genes from HVG selection.

- pct_counts_hb — hemoglobin fraction. 
  - This flags red-blood-cell contamination / ambient hemoglobin. 
  - In a solid-tumor dissociation like PDAC there's almost always blood contamination, 
  - so a high-hb population is usually erythrocytes or ambient-RNA-contaminated cells you want to drop rather than a real cell type of interest.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 8))

for ax, key in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"]):
    sc.pl.violin(ad, key, jitter=0.4, ax=ax, show=False)
plt.tight_layout()



###  it is useful to consider QC metrics jointly by inspecting a scatter plot colored by pct_counts_mt.

In [ ]:
sc.pl.scatter(ad, "total_counts", "n_genes_by_counts", color="pct_counts_mt", show=False);

In [ ]:
from libs.matplotlib_figure_style import *

In [ ]:
apply_figure_style()

fig = plt.figure(figsize=(11, 6.4))
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.32)

ax = fig.add_subplot(gs[0,0])
ax.hist(ad.obs.total_counts, bins=45, color="#3b6ea5", edgecolor="white", linewidth=.3)
ax.axvline(500, color="#c1272d", ls="--", lw=1)
ax.set_xlabel("UMI counts per spot"); ax.set_ylabel("Spots")
ax.set_title("Sequencing depth is uniformly high", loc="left")
ax.annotate("filter: 500", xy=(500,0), xytext=(2600, ax.get_ylim()[1]*.72),
            fontsize=6, color="#c1272d", arrowprops=dict(arrowstyle="-", color="#c1272d", lw=.7))
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x,p: f"{x/1000:.0f}k" if x else "0"))
panel_letter(ax,"a")

ax = fig.add_subplot(gs[0,1])
ax.hist(ad.obs.n_genes_by_counts, bins=45, color="#3b6ea5", edgecolor="white", linewidth=.3)
ax.axvline(250, color="#c1272d", ls="--", lw=1)
ax.set_xlabel("Genes detected per spot"); ax.set_ylabel("Spots")
ax.set_title("Gene complexity centred near 3.8k", loc="left")
panel_letter(ax,"b")

ax = fig.add_subplot(gs[0,2])
sca = ax.scatter(ad.obs.total_counts, ad.obs.n_genes_by_counts, c=ad.obs.pct_counts_mt,
                 s=5, cmap="viridis", linewidths=0)
cb = fig.colorbar(sca, ax=ax, pad=.02); cb.set_label("Mitochondrial reads (%)", fontsize=6)
cb.ax.tick_params(labelsize=5)
ax.set_xlabel("UMI counts"); ax.set_ylabel("Genes detected")
ax.set_title("No depth\u2013mitochondrial coupling", loc="left")
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x,p: f"{x/1000:.0f}k" if x else "0"))
panel_letter(ax,"c")

for j,(col,lab,ttl) in enumerate([("pct_counts_mt","Mitochondrial reads (%)","Mitochondrial fraction is low"),
                                  ("pct_counts_ribo","Ribosomal reads (%)","Ribosomal fraction is stable"),
                                  ("pct_counts_hb","Haemoglobin reads (%)","Blood contamination is minimal")]):
    ax = fig.add_subplot(gs[1,j])
    v = ad.obs[col].values
    ax.hist(v, bins=40, color="#5c8a3c" if j==1 else ("#7b5aa6" if j==2 else "#c98726"),
            edgecolor="white", linewidth=.3)
    ax.set_xlabel(lab); ax.set_ylabel("Spots"); ax.set_title(ttl, loc="left")
    ax.annotate(f"median {np.median(v):.1f}%", xy=(.97,.9), xycoords="axes fraction",
                ha="right", fontsize=6, color=META_GREY)
    panel_letter(ax,"def"[j])

fig.suptitle("Visium QC — HDCA developing human heart, section V10A13-157_A1 (PCW 8)",
             x=.008, ha="left", fontsize=9)
fig.text(.008, .003, "n = 1,070 spots under tissue, 20,751 genes detected in \u22651 spot. "
         "Dashed lines mark applied filters (\u2265500 UMI, \u2265250 genes). Space Ranger filtered matrix.",
         fontsize=6, color=META_GREY)
fig.savefig("qc_visium.png", dpi=300, bbox_inches="tight")
print("saved")


In [ ]:
rho,p = spearmanr(ad.obs.total_counts, ad.obs.pct_counts_mt)
print("spearman(total_counts, pct_mt) rho=%.3f p=%.2g" % (rho,p))

### remove cells that have too many ...

Based on the QC metric plots, one could now remove cells that have too many mitochondrial genes expressed or too many total counts by setting manual or automatic thresholds. However, sometimes what appears to be poor QC metrics can be driven by real biology so we suggest starting with a very permissive filtering strategy and revisiting it at a later point. We therefore now only filter cells with less than 100 genes expressed and genes that are detected in less than 3 cells.

### batches are groups (cases x controls, e.g.)
Additionally, it is important to note that for datasets with multiple batches, quality control should be performed for each sample individually as quality control thresholds can vary substantially between batches.



In [ ]:
ad_raw = ad.copy()

# QC filters
sc.pp.filter_cells(ad, min_counts=500)
sc.pp.filter_cells(ad, min_genes=250)

ad = ad[ad.obs.pct_counts_mt < 25].copy()
sc.pp.filter_genes(ad, min_cells=3)
print("after QC:", ad.shape)

ad.layers["counts"] = ad.X.copy()
sc.pp.normalize_total(ad, target_sum=1e4)
sc.pp.log1p(ad)
ad.raw = ad
sc.pp.highly_variable_genes(ad, n_top_genes=2000, flavor="seurat")
print("HVGs:", int(ad.var.highly_variable.sum()))

sc.pp.scale(ad, max_value=10)
sc.tl.pca(ad, n_comps=50, svd_solver="arpack", mask_var="highly_variable")
var_ratio = ad.uns["pca"]["variance_ratio"]
print("PC1-5 var%:", np.round(var_ratio[:5]*100,2), "| cum 30 PCs: %.1f%%" % (var_ratio[:30].sum()*100))



### Are PC1-5 var% to low? How to evaluate them?

PC1-5 var%: [5.4 2.24 1.43 1.05 0.84] | cum 30 PCs: 21.3%




### pairwise ARI between settings — no privileged reference

Mean Pairwise ARI (Adjusted Rand Index) is a metric used to evaluate the reproducibility, stability, or consensus of unsupervised clustering results. [1] (https://www.researchgate.net/figure/Representation-stability-a-Within-representation-ARI-light-bars-10-kmeans-seeds-and_fig4_411810727), [2] (https://www.mdpi.com/2076-3417/16/9/4137)While a standard ARI compares a single clustering partition against a single "ground truth" set of labels, the mean pairwise ARI is calculated by running a clustering algorithm multiple times (or across different subsets/methods), calculating the ARI for every possible pair of those results, and taking the mathematical average.

https://en.wikipedia.org/wiki/Rand_index


#### 💡 Why is it used?

In real-world data science, you rarely have perfect ground-truth labels. 
To figure out if your clusters are meaningful or just random noise, you measure their stability

Mean pairwise ARI answers the question: 

“If I run this algorithm again with a different random seed, different initialization, or slightly different data, will it give me the same clusters?” 

[1] (https://www.sciencedirect.com/science/article/abs/pii/S2352648322000204), [2] (https://www.mdpi.com/2673-9909/6/2/32)

#### 📊 Understanding the Scores

Because it is based on the Adjusted Rand Index, the score accounts for random chance: 

[1] (https://search.r-project.org/CRAN/refmans/CommKern/help/adj_RI.html), [2] (https://en.wikipedia.org/wiki/Rand_index)

- 1.0: Perfect stability. Every single run produced the exact same cluster assignments.
- \> 0.80: High stability/reproducibility. The algorithm is finding highly reliable structures in the data.
- 0.40 – 0.70: Moderate stability. The clusters are somewhat reliable, but sensitive to tuning parameters or initializations.
- Close to 0.0 (or negative): Poor stability. The algorithm is essentially guessing randomly each time, meaning the chosen number of clusters (K) or features might be invalid.

[1] (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html), [2] (https://iovs.arvojournals.org/article.aspx?articleid=2817988), [3] (https://www.gdc-docs.ethz.ch/UniBS/MultivariateStatistics/site/clustering/), [4] (https://www.frontiersin.org/journals/plant-science/articles/10.3389/fpls.2026.1908413/full), [5] (https://stats.stackexchange.com/questions/416687/clustering-use-ari-to-compare-different-clustering), [6] (https://www.mdpi.com/2076-3417/16/9/4137)




In [ ]:
"""HVG sweep — drop-in. Run AFTER your QC + normalize + log1p cells.

`base` is a clean, log1p-normalised AnnData with NO highly_variable column.
Each sweep iteration needs a fresh copy because sc.pp.scale() overwrites .X
in place (and densifies it), so a reused object would compound across runs.
"""
from sklearn.metrics import adjusted_rand_score as ari
import warnings
warnings.filterwarnings("ignore")


# ---- define `base` -------------------------------------------------------
# If you have already run sc.pp.scale(ad, ...), `ad.X` is z-scored and unusable
# for HVG selection. Recover the log1p state from .raw:
base = ad.raw.to_adata()            # log1p-normalised, sparse, no HVG col

# Equivalent if .raw was never set, rebuilding from the counts layer:
# base = sc.AnnData(ad.layers["counts"].copy(), obs=ad.obs.copy(), var=ad.var[["gene_ids"]].copy())
# base.obsm["spatial"] = ad.obsm["spatial"].copy()
# sc.pp.normalize_total(base, target_sum=1e4); sc.pp.log1p(base)

# ---- sweep ---------------------------------------------------------------
GRID = [500, 1000, 2000, 3000, 5000, 8000]
MARKERS = ["PITX2","BMP10","MYL4","NPPA","MYH7","MYL2","HEY2","TNNI3","IRX3","IRX5",
           "NMRK2","DCN","OGN","ASPN","MGP","COL12A1","POSTN","HAPLN1","SOX9","ELN",
           "ACTA2","NR2F2","KCNA5","ANGPT1"]
MARKERS = [g for g in MARKERS if g in base.var_names]

labels, hvgsets, rows = {}, {}, []
for n in GRID:
    a = base.copy()                                  # <- the isolation step
    sc.pp.highly_variable_genes(a, n_top_genes=n, flavor="seurat")
    hvgsets[n] = set(a.var_names[a.var.highly_variable])
    sc.pp.scale(a, max_value=10)                     # overwrites a.X, densifies
    sc.tl.pca(a, n_comps=50, svd_solver="arpack", mask_var="highly_variable")
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(a, resolution=0.5, flavor="igraph", n_iterations=2,
                 key_added="L", random_state=0)
    labels[n] = a.obs["L"].astype(str).values
    rows.append(dict(n_top_genes=n,
                     n_clusters=len(set(labels[n])),
                     cum_var_30pcs_pct=round(a.uns["pca"]["variance_ratio"][:30].sum()*100, 2),
                     markers_captured=sum(g in hvgsets[n] for g in MARKERS)))
    del a                                            # 151 MB dense once scaled

# Stability = mean pairwise ARI to every OTHER setting.
# Do NOT score against labels from one chosen setting: that setting scores 1.0
# by construction and the comparison is circular.
for r in rows:
    n = r["n_top_genes"]
    r["mean_pairwise_ARI"] = round(
        float(np.mean([ari(labels[n], labels[m]) for m in GRID if m != n])), 3)

sweep = pd.DataFrame(rows).set_index("n_top_genes")
print(sweep.to_string())

In [ ]:
from matplotlib.ticker import NullLocator, NullFormatter

fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.plot(sweep.index, sweep.mean_pairwise_ARI, "-o", color="#888888", ms=4, lw=1.1)
ax.plot([2000], [sweep.loc[2000, "mean_pairwise_ARI"]], "o", color="#c1272d", ms=7)
ax.set_xscale("log")
ax.set_xticks(sweep.index); ax.set_xticklabels(["500","1k","2k","3k","5k","8k"])
ax.xaxis.set_minor_locator(NullLocator())
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("n_top_genes"); ax.set_ylabel("mean pairwise ARI")

### Doublet detection

As a next step, we run a doublet detection algorithm. Identifying doublets is crucial as they can lead to misclassifications or distortions in downstream analysis steps. Scanpy contains the doublet detection method Scrublet [Wolock et al., 2019]. Scrublet predicts cell doublets using a nearest-neighbor classifier of observed transcriptomes and simulated doublets. scanpy.pp.scrublet() adds doublet_score and predicted_doublet to .obs. One can now either filter directly on predicted_doublet or use the doublet_score later during clustering to filter clusters with high doublet scores.

scanpy.pp.scrublet()

Predict cell doublets using a nearest-neighbor classifier of observed transcriptomes and simulated doublets. Works best if the input is a raw (unnormalized) counts matrix from a single sample or a collection of similar samples from the same experiment. This function is a wrapper around functions that pre-process using Scanpy and directly call functions of Scrublet(). You may also undertake your own preprocessing, simulate doublets with scrublet_simulate_doublets(), and run the core scrublet function scrublet() with adata_sim set.

https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.scrublet.html#scanpy.pp.scrublet

In [ ]:
# forbiden
# sc.pp.scrublet(adata, batch_key="sample")

### Do we need in Visium?

It's absent because it doesn't apply to Visium. Two independent reasons, both checkable.

Every spot is already a multi-cell mixture. A Visium spot is 55 µm across — 2,376 µm²,
  - which at fetal cardiomyocyte sizes holds roughly 15–33 cells. 
  
Scrublet's question is "is this barcode one cell or two?" 
  - On Visium the answer is always "neither, it's ~20." There's no doublet to detect because there's no singlet baseline. A spot is a small tissue volume, not a cell.

batch_key="sample" has nothing to bind to. I checked: 'sample' isn't in ad.obs, no batch-like column exists, and this object holds one section (len(uns['spatial']) == 1). That call would raise a KeyError. The parameter only makes sense once you have several sections concatenated — which is the PCW 5.5–14 series, not this one.

I ran it regardless, because "inapplicable" is more convincing when demonstrated. sc.pp.scrublet exists in scanpy 1.11.5 and completes without error — it calls 1 spot of 1,067 a doublet at threshold 0.34. But look at what the score tracks:

| Domain | Median | Mean | n spots |
|---|---|---|---|
| Left atrial myocardium | 0.032 | 0.043 | 119 |
| Right atrial myocardium | 0.032 | 0.038 | 107 |
| Trabecular ventricular myocardium | 0.063 | 0.073 | 265 |
| Valve mesenchyme & vessel wall | 0.063 | 0.068 | 79 |
| Compact ventricular myocardium | 0.070 | 0.075 | 343 |
| Fibrous–interstitial myocardium | 0.070 | 0.082 | 154 |

The score is domain-structured, and correlates 0.34 with total counts and 0.35 with gene count. It's measuring transcriptional complexity — which on Visium means cellular heterogeneity within the spot. Dense compact myocardium and mixed fibrous tissue score high; atrial spots score low. Running it and filtering would preferentially delete your most cell-dense regions. That's an active harm, not a no-op.

The Visium equivalent of the question Scrublet answers is deconvolution: not "remove mixed spots" but "estimate the cell-type proportions within each spot." The HDCA paper uses stereoscope against matched scRNA-seq for exactly this, and it's one of the follow-ups I flagged earlier.

Where Scrublet would belong in this project is the scRNA-seq side — HDCA_heart_SC_annotations_HL_240115.csv in your metadata implies dissociated single-cell data, and there batch_key pointing at the per-donor or per-library column is correct and worth doing.

### UMAP -> Leiden

In [ ]:
# Compute the nearest neighbors distance matrix and a neighborhood graph of observations :cite:p:`McInnes2018`.
sc.pp.neighbors(ad, n_neighbors=15, n_pcs=30)
sc.tl.umap(ad, random_state=0)

for res in [0.3, 0.5, 0.8, 1.0]:
    sc.tl.leiden(ad, resolution=res, key_added=f"leiden_{res}", flavor="igraph", n_iterations=2, directed=False, random_state=0)
    print(f"res {res}: {ad.obs[f'leiden_{res}'].nunique():<5} clusters  sizes={sorted(ad.obs[f'leiden_{res}'].value_counts().tolist(), reverse=True)}")


In [ ]:
DOMAIN = {
    "0": "Right atrial myocardium",
    "1": "Compact ventricular myocardium",
    "2": "Fibrous–interstitial myocardium",
    "3": "Trabecular ventricular myocardium",
    "4": "Left atrial myocardium",
    "5": "Valve mesenchyme & vessel wall",
}
SHORT = {"Right atrial myocardium": "RA", "Compact ventricular myocardium": "V_C",
         "Fibrous–interstitial myocardium": "FI", "Trabecular ventricular myocardium": "V_T",
         "Left atrial myocardium": "LA", "Valve mesenchyme & vessel wall": "VM/TM"}

ad.obs["domain"] = ad.obs["leiden_0.5"].map(DOMAIN).astype("category")
ad.obs["domain_short"] = ad.obs["domain"].map(SHORT).astype("category")

In [ ]:
ad.obs["domain_short"]

In [ ]:
"""Leiden resolution diagnostics for a single Visium section.

Builds the two DataFrames used in the resolution figure:
  coh  -- spatial coherence, observed vs label-permutation null
  rdf  -- one row per resolution: k, clusters with marker support,
          subsample ARI, smallest cluster, spatial coherence

Input `A` is the processed AnnData (1067 x 17673) with:
  A.obsm["X_pca"]        50 comps on 2000 scaled HVGs
  A.raw                  log1p-normalised counts (use_raw=True for markers)
  A.obs["domain_short"]  the six res-0.5 domain labels
  A.obsm["spatial"]      Visium array coordinates
"""
from sklearn.metrics import adjusted_rand_score

RES = [0.3, 0.5, 0.8, 1.0, 1.4]
rng = np.random.default_rng(0)

# ---- 1. sweep on a fresh neighbour graph -------------------------------
r = ad.copy()
sc.pp.neighbors(r, n_neighbors=15, n_pcs=30)
sc.tl.umap(r, random_state=0)
for res in RES:
    sc.tl.leiden(r, resolution=res, key_added=f"L{res}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)

# ---- 2. spatial coherence vs permutation null --------------------------
sq.gr.spatial_neighbors(r, coord_type="grid", n_neighs=6)
Wsp = r.obsp["spatial_connectivities"].tocsr()
ii, jj = Wsp.nonzero()
keep = ii < jj                      # each hex edge once
ii, jj = ii[keep], jj[keep]

def spatial_coherence(lab):
    lab = np.asarray(lab)
    return float((lab[ii] == lab[jj]).mean())

rows = []
for res in RES:
    lab = r.obs[f"L{res}"].values
    obs = spatial_coherence(lab)
    null = np.mean([spatial_coherence(rng.permutation(lab)) for _ in range(20)])
    rows.append(dict(res=res, observed=obs, null=null, ratio=obs / null))
coh = pd.DataFrame(rows)

# ---- 3. marker support: clusters with >=3 genes p_adj<1e-3 & logFC>1 ---
def n_supported(res):
    tmp = r.copy()
    tmp.obs["L"] = tmp.obs[f"L{res}"]
    sc.tl.rank_genes_groups(tmp, "L", method="wilcoxon", use_raw=True)
    n = 0
    for g in tmp.obs["L"].cat.categories:
        d = sc.get.rank_genes_groups_df(tmp, group=g)
        if ((d.pvals_adj < 1e-3) & (d.logfoldchanges > 1)).sum() >= 3:
            n += 1
    return n

# ---- 4. subsample stability: 5 seeds x 80% of spots -------------------
def subsample_ari(res, n_seeds=5, frac=0.8):
    ref = r.obs[f"L{res}"].astype(str).values
    out = []
    for s in range(n_seeds):
        idx = np.random.default_rng(s).choice(r.n_obs, int(frac * r.n_obs), replace=False)
        sub = r[np.sort(idx)].copy()
        sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
        sc.tl.leiden(sub, resolution=res, key_added="Ls",
                     flavor="igraph", n_iterations=2, directed=False, random_state=0)
        out.append(adjusted_rand_score(ref[np.sort(idx)], sub.obs["Ls"].astype(str).values))
        del sub
    return float(np.mean(out))

rdf = pd.DataFrame([
    dict(res=res,
         k=r.obs[f"L{res}"].nunique(),
         clusters_with_markers=n_supported(res),
         subsample_ARI=round(subsample_ari(res), 3),
         min_size=int(r.obs[f"L{res}"].value_counts().min()),
         spatial_coherence=round(coh.loc[coh.res == res, "observed"].item(), 3))
    for res in RES
])

# ---- 5. what the extra clusters at res 0.8 are ------------------------
ct = pd.crosstab(r.obs["L0.8"], r.obs["domain_short"])
ct["size"] = ct.sum(axis=1)
ct["purity"] = (ct.iloc[:, :-1].max(axis=1) / ct["size"]).round(2)

print(rdf.to_string(index=False))
print(ct.to_string())

In [ ]:
C='navy'
G='green'

import libs.matplotlib_figure_style as fs

# fix the label collision in panel c
fig, axes = plt.subplots(1, 3, figsize=(7.8, 2.4))
rs = rdf.res.values
ax = axes[0]
ax.plot(rs, rdf.spatial_coherence, "-o", color=C, ms=4, lw=1.1, label="observed")
ax.plot(rs, coh.null.values, "--", color=G, lw=1.0, label="permuted null")
ax.set_xlabel("Leiden resolution"); ax.set_ylabel("fraction of same-label\nspatial neighbours")
ax.set_ylim(0, 1); ax.legend(loc="upper right")
ax.set_title("Domains stay spatially contiguous", loc="left", fontsize=7.5, pad=6)
ax = axes[1]
ax.plot(rs, rdf.subsample_ARI, "-o", color=C, ms=4, lw=1.1)
ax.set_xlabel("Leiden resolution"); ax.set_ylabel("ARI, 80% subsample")
ax.set_ylim(0.5, 0.75)
ax.set_title("Stability is flat across 0.3\u20131.4", loc="left", fontsize=7.5, pad=6)
ax = axes[2]
ax.bar(rs, rdf.min_size, width=0.09, color=[C if m < 50 else G for m in rdf.min_size])
ax.axhline(30, ls=":", lw=0.7, color=G)
ax.annotate("30 spots", xy=(1.42, 30), xytext=(0, 4), textcoords="offset points",
            fontsize=6, color=G, ha="right", va="bottom")
ax.set_xlabel("Leiden resolution"); ax.set_ylabel("smallest cluster (spots)")
ax.set_ylim(0, 100)
ax.set_title("Small clusters appear from 0.8", loc="left", fontsize=7.5, pad=6)
for a, l in zip(axes, "abc"): fs.panel_letter(a, l, dx=-0.26, dy=1.12)
fig.suptitle("Resolution 0.5 for chamber anatomy; 0.8 additionally resolves a 20-spot macrophage cluster",
             y=1.06, x=0.0, ha="left", fontsize=8.5)
fig.tight_layout(w_pad=2.4)
fig.savefig("leiden_resolution.png", dpi=300, bbox_inches="tight")
print(coh.round(3).to_string(index=False))

In [ ]:
KEY="leiden_0.5"
sc.tl.rank_genes_groups(ad, KEY, method="wilcoxon", pts=True)
top = pd.DataFrame({g: [ad.uns["rank_genes_groups"]["names"][g][i] for i in range(18)]
                    for g in ad.obs[KEY].cat.categories})
print("=== top 18 markers per cluster ==="); print(top.to_string())

panel = {
 "Compact myocardium":["MYH7","TNNT2","ACTC1","MYL2"],
 "Trabecular/vCM":["NPPA","MYL7","BMP10"],
 "Atrial CM":["NR2F2","KCNA5","MYL4","SLN"],
 "Endocardium":["NPR3","EGFL7","PECAM1","CDH5"],
 "Epicardium":["WT1","UPK3B","BNC1","TBX18"],
 "Valve/cushion mesenchyme":["POSTN","SOX9","COL1A1","TNC","HAPLN1"],
 "Conduction":["SHOX2","TBX3","HCN4"],
 "Smooth muscle / vessel":["ACTA2","TAGLN","MYH11","ELN"],
 "Blood":["HBB","HBA1","HBG1"],
}
avail={k:[g for g in v if g in ad.raw.var_names] for k,v in panel.items()}
sc.tl.dendrogram(ad, groupby=KEY)
mm = sc.get.obs_df(ad, keys=[g for v in avail.values() for g in v]+[KEY], use_raw=True)
print("\n=== mean log-expression by cluster ===")
print(mm.groupby(KEY, observed=True).mean().round(2).to_string())


In [ ]:
sc.tl.rank_genes_groups(ad, KEY, groups=["0"], reference="4", method="wilcoxon", key_added="c0_vs_c4")
n=ad.uns["c0_vs_c4"]["names"]["0"]; l=ad.uns["c0_vs_c4"]["logfoldchanges"]["0"]
print("0 vs 4 UP:", [(n[i], round(float(l[i]),2)) for i in range(12)])
print("0 vs 4 DOWN:", [(n[-i-1], round(float(l[-i-1]),2)) for i in range(12)])

extra=["PITX2","NR2F2","KCNJ3","ANGPT1","HEY2","IRX3","IRX5","MYH11","BMP10","NPPA","SLN","KCNA5","WT1","TBX18","POSTN","DCN","OGN","LUM","ELN","SOX9","TNNI3","MYL2","MB","NMRK2","CNN1","RGS5","PDGFRB","HCN4","SHOX2","TBX3","CAV1","LYVE1","PTPRC"]
av=[g for g in extra if g in ad.raw.var_names]
d=sc.get.obs_df(ad, keys=av+[KEY], use_raw=True).groupby(KEY, observed=True).mean().round(2)
print("\n", d.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
sq.pl.spatial_scatter(ad, color=KEY, size=1.35, img_alpha=.85, ax=axes[0], frameon=False, legend_loc="right margin", title="")
sc.pl.umap(ad, color=KEY, ax=axes[1], show=False, frameon=False, title="", legend_loc="on data")
fig.savefig("_check.png", dpi=130, bbox_inches="tight")
print("ok")

In [ ]:
root_meta

In [ ]:
filename = root_meta / "HDCA_heart_ST_annotations.csv"

ann = pd.read_csv(filename, sep=';', index_col=0)
print(ann.shape); print(ann.columns.tolist()[:20]); print(ann.head(3).to_string())

In [ ]:
sets = {
 "V_EP (ventricular epicardium)":["WT1","UPK3B","BNC1","TBX18","KRT19","MSLN","ALDH1A2","SFRP5"],
 "TM/TA (vessel wall)":["ACTA2","TAGLN","MYH11","CNN1","ELN","RGS5","PDGFRB","NOTCH3"],
 "VM (valve mesenchyme)":["POSTN","SOX9","HAPLN1","TNC","CRABP1","COL2A1","ACAN"],
 "EN (endocardium)":["NPR3","EGFL7","PECAM1","CDH5","CLDN5","PLVAP"],
 "AVP":["TBX2","TBX3","RSPO3","BMP2","MSX2"],
 "Compact vCM":["MYH7","MYL2","HEY2","TNNI3","FHL2","MB"],
 "Trabecular vCM":["NPPA","IRX3","IRX5","SLC8A1","NMRK2"],
 "Atrial CM":["MYL4","KCNA5","NR2F2","SLN","MYL7"],
}
for k,v in sets.items():
    g=[x for x in v if x in ad.raw.var_names]
    sc.tl.score_genes(ad, g, score_name=k, use_raw=True)

print(ad.obs.groupby(KEY, observed=True)[list(sets)].mean().round(2).to_string())


In [ ]:
sq.gr.spatial_neighbors(ad, coord_type="grid", n_neighs=6)
deg = np.asarray(ad.obsp["spatial_connectivities"].sum(1)).ravel()
ad.obs["n_nb"] = deg
ad.obs["is_edge"] = deg < 6
print(ad.obs.groupby(KEY, observed=True).agg(mean_nb=("n_nb","mean"), frac_edge=("is_edge","mean")).round(2).to_string())
print("\nglobal frac_edge %.2f" % ad.obs.is_edge.mean())


In [ ]:
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon, Point
xy = ad.obsm["spatial"]
hull = ConvexHull(xy)
poly = Polygon(xy[hull.vertices])
d = np.array([poly.exterior.distance(Point(p)) for p in xy])
sf = ad.uns["spatial"][list(ad.uns["spatial"])[0]]["scalefactors"]
ad.obs["dist_surface_um"] = d / sf["spot_diameter_fullres"] * 55.0
print(ad.obs.groupby(KEY, observed=True)["dist_surface_um"].describe()[["mean","50%"]].round(0).to_string())
print("\nglobal median %.0f um" % ad.obs.dist_surface_um.median())


In [ ]:
from scipy.ndimage import distance_transform_edt, binary_fill_holes, binary_closing
ar = ad.obs["array_row"].values.astype(int); ac = ad.obs["array_col"].values.astype(int)
H, W = ar.max()+3, ac.max()+3
m = np.zeros((H, W), bool); m[ar+1, ac+1] = True
mf = binary_fill_holes(binary_closing(m, np.ones((3,3), bool)))
dt = distance_transform_edt(mf)
ad.obs["depth_lat"] = dt[ar+1, ac+1]
print(ad.obs.groupby(KEY, observed=True)["depth_lat"].agg(["mean","median"]).round(2).to_string())
print("\nglobal median %.2f lattice units" % ad.obs.depth_lat.median())
print("\nfrac of spots at depth<=1 (outer surface) per cluster:")
print(ad.obs.assign(surf=ad.obs.depth_lat<=1).groupby(KEY, observed=True)["surf"].mean().round(2).to_string())


In [ ]:
labels = {"0":"Right atrial myocardium","1":"Compact ventricular myocardium",
          "2":"Fibrous–interstitial myocardium","3":"Trabecular ventricular myocardium",
          "4":"Left atrial myocardium","5":"Valve mesenchyme & vessel wall"}
short  = {"0":"RA","1":"V_C","2":"FI","3":"V_T","4":"LA","5":"VM/TM"}
ad.obs["domain"] = ad.obs[KEY].map(labels).astype("category")
ad.obs["domain_short"] = ad.obs[KEY].map(short).astype("category")
order = ["Left atrial myocardium","Right atrial myocardium","Compact ventricular myocardium",
         "Trabecular ventricular myocardium","Fibrous–interstitial myocardium","Valve mesenchyme & vessel wall"]
ad.obs["domain"] = ad.obs["domain"].cat.reorder_categories(order)

sq.gr.spatial_autocorr(ad, mode="moran", genes=ad.var_names[ad.var.highly_variable].tolist(), n_perms=100, n_jobs=8)
mor = ad.uns["moranI"]
print("=== top 25 spatially variable genes (Moran's I) ===")
print(mor.head(25)[["I","pval_norm_fdr_bh"]].round(4).to_string())


In [ ]:
sq.gr.spatial_autocorr(ad, mode="moran", genes=ad.var_names[ad.var.highly_variable].tolist(), n_perms=None, n_jobs=1)
mor = ad.uns["moranI"]
print("=== top 30 spatially variable genes (Moran's I) ===")
print(mor.head(30)[["I","pval_norm_fdr_bh"]].round(4).to_string())


In [ ]:
PAL = {"Left atrial myocardium":"#7b5aa6","Right atrial myocardium":"#3b6ea5",
       "Compact ventricular myocardium":"#c98726","Trabecular ventricular myocardium":"#c1272d",
       "Fibrous–interstitial myocardium":"#5c8a3c","Valve mesenchyme & vessel wall":"#8c5a3b"}
ad.uns["domain_colors"] = [PAL[c] for c in ad.obs["domain"].cat.categories]

fig = plt.figure(figsize=(12.4, 8.6))
gs = fig.add_gridspec(2, 2, height_ratios=[1, .92], hspace=.3, wspace=.14)

axA = fig.add_subplot(gs[0,0])
sc.pl.umap(ad, color="domain", ax=axA, show=False, frameon=False, title="", s=26, legend_loc=None)
axA.set_title("Transcriptional domains separate by chamber and wall layer", loc="left", fontsize=8)
panel_letter(axA,"a")

axB = fig.add_subplot(gs[0,1])
sq.pl.spatial_scatter(ad, color="domain", size=1.45, img_alpha=.9, ax=axB, frameon=False,
                      title="", legend_loc=None, palette=PAL)
axB.set_title("The same domains reconstruct heart anatomy in situ", loc="left", fontsize=8)
panel_letter(axB,"b")

hs=[plt.Line2D([0],[0],marker="o",ls="",mfc=PAL[c],mec="none",ms=6) for c in order]
axB.legend(hs, [f"{c}  (n={int((ad.obs.domain==c).sum())})" for c in order],
           loc="upper left", bbox_to_anchor=(-1.12,-.02), ncol=3, frameon=False,
           fontsize=6.5, handletextpad=.4, columnspacing=1.2)

axC = fig.add_subplot(gs[1,:])
mk = ["MYL4","KCNA5","NR2F2","PITX2","BMP10","ANGPT1","MYH7","MYL2","HEY2","TNNI3",
      "NPPA","IRX3","NMRK2","DCN","OGN","MGP","COL1A1","POSTN","HAPLN1","ELN","ACTA2","RGS5"]
mk = [g for g in mk if g in ad.raw.var_names]
sub = ad[ad.obs.sort_values("domain").index]
sc.pl.matrixplot(sub, mk, groupby="domain", use_raw=True, standard_scale="var",
                 cmap="RdBu_r", ax=axC, show=False, colorbar_title="Scaled mean\nexpression",
                 swap_axes=False)
panel_letter(axC,"c", dx=-.055)
axC.set_title("Chamber- and layer-specific markers confirm each domain's identity", loc="left", fontsize=8)

fig.suptitle("Spatial domains of the PCW 8 human heart — Visium section V10A13-157_A1", x=.008, ha="left", fontsize=10)
fig.text(.008,-.045,"n = 1,067 spots after QC. Leiden clustering (res 0.5) on 30 PCs of 2,000 HVGs. "
        "Panel c: mean log-normalised expression per domain, scaled 0\u20131 per gene. Labels assigned from marker "
        "expression and in-situ position; domain names follow the HDCA annotation vocabulary.",
        fontsize=6, color=META_GREY)
fig.savefig("umap_domains.png", dpi=300, bbox_inches="tight")
print("saved")


In [ ]:
fig = plt.figure(figsize=(12.4, 9.0))
gs = fig.add_gridspec(2, 2, height_ratios=[1, .82], hspace=.34, wspace=.12)

axA = fig.add_subplot(gs[0,0])
sc.pl.umap(ad, color="domain", ax=axA, show=False, frameon=False, title="", s=26, legend_loc=None)
axA.set_title("Transcriptional domains separate by chamber and wall layer", loc="left", fontsize=8)
panel_letter(axA,"a")

axB = fig.add_subplot(gs[0,1])
sq.pl.spatial_scatter(ad, color="domain", size=1.45, img_alpha=.9, ax=axB, frameon=False,
                      title="", legend_loc=None)
axB.set_title("The same domains reconstruct heart anatomy in situ", loc="left", fontsize=8)
panel_letter(axB,"b")

hs=[plt.Line2D([0],[0],marker="o",ls="",mfc=PAL[c],mec="none",ms=6) for c in order]
axB.legend(hs, [f"{c}  (n={int((ad.obs.domain==c).sum())})" for c in order],
           loc="upper left", bbox_to_anchor=(-1.14,-.03), ncol=3, frameon=False,
           fontsize=6.5, handletextpad=.4, columnspacing=1.4)

# panel c: manual scaled-mean heatmap
groups = {"Atrial":["MYL4","KCNA5","NR2F2","PITX2","BMP10","ANGPT1"],
          "Ventricular":["MYH7","MYL2","HEY2","TNNI3","NPPA","IRX3","NMRK2"],
          "Stromal / vessel":["DCN","OGN","MGP","COL1A1","POSTN","HAPLN1","ELN","ACTA2","RGS5"]}
mk=[g for v in groups.values() for g in v if g in ad.raw.var_names]
M = sc.get.obs_df(ad, keys=mk+["domain"], use_raw=True).groupby("domain", observed=True).mean().loc[order]
Ms = (M - M.min()) / (M.max() - M.min())

axC = fig.add_subplot(gs[1,:])
im = axC.imshow(Ms.values, cmap="RdBu_r", aspect="auto", vmin=0, vmax=1)
axC.set_xticks(range(len(mk))); axC.set_xticklabels(mk, rotation=90, fontsize=6.5, style="italic")
axC.set_yticks(range(len(order))); axC.set_yticklabels(order, fontsize=7)
for c in [PAL[o] for o in order]: pass
for i,o in enumerate(order):
    axC.add_patch(plt.Rectangle((-1.9,i-.5),.75,1,color=PAL[o],clip_on=False,lw=0))
axC.set_xlim(-.5,len(mk)-.5)
cb = fig.colorbar(im, ax=axC, pad=.012, fraction=.022); cb.set_label("Scaled mean expression", fontsize=6.5)
cb.ax.tick_params(labelsize=5.5)
x=0
for gname,gl in groups.items():
    gl=[g for g in gl if g in mk]; 
    axC.plot([x-.45,x+len(gl)-.55],[-.78,-.78],color=META_GREY,lw=.8,clip_on=False)
    axC.annotate(gname, xy=(x+len(gl)/2-.5,-1.05), ha="center", fontsize=6.5, color=META_GREY, annotation_clip=False)
    x+=len(gl)
for s in axC.spines.values(): s.set_visible(False)
axC.tick_params(length=0)
axC.set_title("Chamber- and layer-specific markers confirm each domain's identity", loc="left", fontsize=8, pad=18)
panel_letter(axC,"c", dx=-.075)

fig.suptitle("Spatial domains of the PCW 8 human heart — Visium section V10A13-157_A1", x=.008, ha="left", fontsize=10)
fig.text(.008,-.055,"n = 1,067 spots after QC. Leiden clustering (resolution 0.5) on 30 PCs of 2,000 highly variable genes. "
        "Panel c: mean log-normalised expression per domain, min\u2013max scaled per gene. Labels assigned from marker expression "
        "and in-situ position; names follow the HDCA annotation vocabulary (Zenodo 15912657).",
        fontsize=6, color=META_GREY)
fig.savefig("umap_domains.png", dpi=300, bbox_inches="tight")
print("saved", Ms.shape)


In [ ]:
sf_h = sf["tissue_hires_scalef"]
xs, ys = xy[:,0]*sf_h, xy[:,1]*sf_h
pad = 40
XL = (xs.min()-pad, xs.max()+pad); YL = (ys.max()+pad, ys.min()-pad)
print("hires img", ad.uns["spatial"][list(ad.uns["spatial"])[0]]["images"]["hires"].shape)
print("x", np.round(XL,1), "y", np.round(YL,1))

In [ ]:
fig = plt.figure(figsize=(12.4, 9.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, .78], hspace=.30, wspace=.10)

axA = fig.add_subplot(gs[0,0])
sc.pl.umap(ad, color="domain", ax=axA, show=False, frameon=False, title="", s=26, legend_loc=None)
axA.set_title("Transcriptional domains separate by chamber and wall layer", loc="left", fontsize=8)
panel_letter(axA,"a")

axB = fig.add_subplot(gs[0,1])
sq.pl.spatial_scatter(ad, color="domain", size=1.45, img_alpha=.9, ax=axB, frameon=False,
                      title="", legend_loc=None)
axB.set_xlim(*XL); axB.set_ylim(*YL)
axB.set_title("The same domains reconstruct heart anatomy in situ", loc="left", fontsize=8)
panel_letter(axB,"b")
sb = 500*sf_h/ (sf["spot_diameter_fullres"]*sf_h) * (sf["spot_diameter_fullres"]*sf_h)/55*0  # placeholder
px_per_um = sf_h * sf["spot_diameter_fullres"] / 55.0
L = 500*px_per_um
x0,y0 = XL[0]+60, YL[0]-70
axB.plot([x0,x0+L],[y0,y0], color="black", lw=2, solid_capstyle="butt")
axB.annotate("500 µm", xy=(x0+L/2, y0-22), ha="center", fontsize=6, color="black")

for i,c in enumerate(order):
    axB.annotate(f"{short[[k for k,v in labels.items() if v==c][0]]}", xy=(0,0), alpha=0)
hs=[plt.Line2D([0],[0],marker="o",ls="",mfc=PAL[c],mec="none",ms=6) for c in order]
axB.legend(hs, [f"{c}  (n={int((ad.obs.domain==c).sum())})" for c in order],
           loc="upper left", bbox_to_anchor=(-1.14,-.03), ncol=3, frameon=False,
           fontsize=6.5, handletextpad=.4, columnspacing=1.4)

axC = fig.add_subplot(gs[1,:])
im = axC.imshow(Ms.values, cmap="RdBu_r", aspect="auto", vmin=0, vmax=1)
axC.set_xticks(range(len(mk))); axC.set_xticklabels(mk, rotation=90, fontsize=6.5, style="italic")
axC.set_yticks(range(len(order))); axC.set_yticklabels(order, fontsize=7)
for i,o in enumerate(order):
    axC.add_patch(plt.Rectangle((-1.9,i-.5),.75,1,color=PAL[o],clip_on=False,lw=0))
axC.set_xlim(-.5,len(mk)-.5)
cb = fig.colorbar(im, ax=axC, pad=.012, fraction=.022); cb.set_label("Scaled mean expression", fontsize=6.5)
cb.ax.tick_params(labelsize=5.5)
x=0
for gname,gl in groups.items():
    gl=[g for g in gl if g in mk]
    axC.plot([x-.45,x+len(gl)-.55],[-.78,-.78],color=META_GREY,lw=.8,clip_on=False)
    axC.annotate(gname, xy=(x+len(gl)/2-.5,-1.05), ha="center", fontsize=6.5, color=META_GREY, annotation_clip=False)
    x+=len(gl)
for s in axC.spines.values(): s.set_visible(False)
axC.tick_params(length=0)
axC.set_title("Chamber- and layer-specific markers confirm each domain's identity", loc="left", fontsize=8, pad=18)
panel_letter(axC,"c", dx=-.075)

fig.suptitle("Spatial domains of the PCW 8 human heart — Visium section V10A13-157_A1", x=.008, ha="left", fontsize=10)
fig.text(.008,-.05,"n = 1,067 spots after QC. Leiden clustering (resolution 0.5) on 30 PCs of 2,000 highly variable genes. "
        "Panel c: mean log-normalised expression per domain, min\u2013max scaled per gene. Labels assigned from marker expression "
        "and in-situ position; names follow the HDCA annotation vocabulary (Zenodo 15912657).",
        fontsize=6, color=META_GREY)
fig.savefig("umap_domains.png", dpi=300, bbox_inches="tight")
print("saved")


In [ ]:
genes = ["NPPA","MYH7","BMP10","PITX2","MYL2","DCN","POSTN","ELN"]
notes = {"NPPA":"trabecular + atrial CM","MYH7":"compact myocardium","BMP10":"right atrium",
         "PITX2":"left atrium","MYL2":"ventricular CM","DCN":"interstitial fibroblasts",
         "POSTN":"valve mesenchyme","ELN":"great-vessel wall"}
I = {g: float(mor.loc[g,"I"]) for g in genes}

fig, axes = plt.subplots(2, 4, figsize=(13.6, 7.2))
for ax, g in zip(axes.ravel(), genes):
    sq.pl.spatial_scatter(ad, color=g, size=1.35, img_alpha=.55, ax=axes.ravel()[0] if False else ax,
                          frameon=False, title="", cmap="magma", use_raw=True, colorbar=True)
    ax.set_xlim(*XL); ax.set_ylim(*YL)
    ax.set_title(f"$\\it{{{g}}}$ — {notes[g]}", loc="left", fontsize=8)
    ax.annotate(f"Moran's I = {I[g]:.2f}", xy=(.98,.02), xycoords="axes fraction", ha="right",
                fontsize=6, color="white")
axes.ravel()[0].plot([XL[0]+60, XL[0]+60+L],[YL[0]-70]*2, color="black", lw=2, solid_capstyle="butt")
axes.ravel()[0].annotate("500 µm", xy=(XL[0]+60+L/2, YL[0]-24), ha="center", fontsize=6)
fig.suptitle("Spatially variable genes on the H&E section — PCW 8 human heart (V10A13-157_A1)", x=.008, ha="left", fontsize=10)
fig.text(.008,.005,"Log-normalised expression per 55 µm Visium spot overlaid on the haematoxylin & eosin image. "
         "Genes are the highest-ranking spatially autocorrelated HVGs (Moran's I, all FDR < 0.001); "
         "colour scales are independent per panel.", fontsize=6, color=META_GREY)
fig.tight_layout(rect=[0,.03,1,.96])
fig.savefig("spatial_genes.png", dpi=300, bbox_inches="tight")
print("saved")


In [ ]:
img = ad.uns["spatial"][list(ad.uns["spatial"])[0]]["images"]["hires"]
spot_pt = (sf["spot_diameter_fullres"]*sf_h)

fig, axes = plt.subplots(2, 4, figsize=(13.6, 7.4))
for ax, g in zip(axes.ravel(), genes):
    v = sc.get.obs_df(ad, keys=[g], use_raw=True)[g].values
    ax.imshow(img)
    z = v <= 0
    ax.scatter(xs[z], ys[z], s=6, c="#d9d9d9", linewidths=0, alpha=.55)
    sca = ax.scatter(xs[~z], ys[~z], s=6, c=v[~z], cmap="magma_r", linewidths=0,
                     vmin=0, vmax=np.percentile(v[~z], 99))
    ax.set_xlim(*XL); ax.set_ylim(*YL); ax.axis("off")
    ax.set_title(f"$\\it{{{g}}}$ — {notes[g]}", loc="left", fontsize=8)
    cb = fig.colorbar(sca, ax=ax, fraction=.042, pad=.015)
    cb.ax.tick_params(labelsize=5.5); cb.outline.set_visible(False)
    cb.set_label("log expr.", fontsize=5.5)
    ax.annotate(f"Moran's I = {I[g]:.2f}   •   {100*(~z).mean():.0f}% spots +",
                xy=(.02,.02), xycoords="axes fraction", fontsize=6, color="#333333",
                bbox=dict(fc="white", ec="none", alpha=.75, pad=1.5))

a0 = axes.ravel()[0]
a0.plot([XL[0]+60, XL[0]+60+L],[YL[0]-110]*2, color="black", lw=2, solid_capstyle="butt")
a0.annotate("500 µm", xy=(XL[0]+60+L/2, YL[0]-150), ha="center", fontsize=6)

fig.suptitle("Spatially variable genes on the H&E section — PCW 8 human heart (V10A13-157_A1)", x=.008, ha="left", fontsize=10)
fig.text(.008,.005,"Log-normalised expression per 55 µm Visium spot overlaid on the haematoxylin & eosin image. Grey spots have zero "
         "detected counts for that gene. Genes are top-ranking spatially autocorrelated HVGs (Moran's I, all FDR < 0.001); "
         "colour scale is per-panel, 0 to the 99th percentile of expressing spots.", fontsize=6, color=META_GREY)
fig.tight_layout(rect=[0,.035,1,.955])
fig.savefig("spatial_genes.png", dpi=300, bbox_inches="tight")
print("saved")


In [ ]:
grey = img.mean(2)
fig, axes = plt.subplots(2, 4, figsize=(13.6, 7.4))
for ax, g in zip(axes.ravel(), genes):
    v = sc.get.obs_df(ad, keys=[g], use_raw=True)[g].values
    ax.imshow(grey, cmap="gray", vmin=grey.min(), vmax=grey.max()*1.02)
    z = v <= 0
    ax.scatter(xs[z], ys[z], s=5.5, facecolors="none", edgecolors="#9e9e9e", linewidths=.25)
    sca = ax.scatter(xs[~z], ys[~z], s=6, c=v[~z], cmap="viridis", linewidths=0,
                     vmin=0, vmax=np.percentile(v[~z], 99))
    ax.set_xlim(*XL); ax.set_ylim(*YL); ax.axis("off")
    ax.set_title(f"$\\it{{{g}}}$ — {notes[g]}", loc="left", fontsize=8)
    cb = fig.colorbar(sca, ax=ax, fraction=.042, pad=.015)
    cb.ax.tick_params(labelsize=5.5); cb.outline.set_visible(False)
    cb.set_label("log expr.", fontsize=5.5)
    ax.annotate(f"Moran's I = {I[g]:.2f}   •   {100*(~z).mean():.0f}% spots detected",
                xy=(.02,.02), xycoords="axes fraction", fontsize=6, color="#222222",
                bbox=dict(fc="white", ec="none", alpha=.8, pad=1.5))

a0 = axes.ravel()[0]
a0.plot([XL[0]+60, XL[0]+60+L],[YL[0]-110]*2, color="black", lw=2, solid_capstyle="butt")
a0.annotate("500 µm", xy=(XL[0]+60+L/2, YL[0]-150), ha="center", fontsize=6)

fig.suptitle("Spatially variable genes on the H&E section — PCW 8 human heart (V10A13-157_A1)", x=.008, ha="left", fontsize=10)
fig.text(.008,.005,"Log-normalised expression per 55 µm Visium spot on greyscale haematoxylin & eosin histology. Open circles mark spots with "
         "zero detected counts. Genes are top-ranking spatially autocorrelated HVGs (Moran's I, all FDR < 0.001); colour scale is "
         "per-panel, 0 to the 99th percentile of expressing spots.", fontsize=6, color=META_GREY)
fig.tight_layout(rect=[0,.035,1,.955])
fig.savefig("spatial_genes.png", dpi=300, bbox_inches="tight")
print("saved")
